# FYP Pipeline - Consolidated Colab Notebook

**How this notebook works:**
- Your DATA (dataset, frames, masks, checkpoints, everything) lives in Google Drive, so it survives disconnects.
- Your CODE ENVIRONMENT (installed packages, cloned repos) resets every time Colab disconnects (~12hr limit / idle timeout), so it must be rebuilt each session.
- **After any disconnect: just run Cell 1 (Setup), then continue from wherever you left off.** Checkpoints only download once ever (cached in Drive); everything else in Cell 1 reruns fast (a couple minutes).

**Before first use:** in your Google Drive, create a folder at `MyDrive/FYP` containing:
- `dataset/real/` and `dataset/fake/` with your videos
- `prompts.json` (from running `annotate_prompts.py` locally beforehand -- this step needs a display window, so it stays local)

**Runtime type:** Runtime -> Change runtime type -> T4 GPU, before running anything.

## Cell 1: SETUP -- run this first every session (and after any reconnect)

In [ ]:
# ============================================================
# SETUP: run this cell every time you (re)connect to Colab.
# Rebuilds the code environment; skips anything already cached in Drive.
# ============================================================
import os

# --- 1. Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/FYP'
assert os.path.isdir(PROJECT_DIR), (
    f'ERROR: {PROJECT_DIR} not found. Create this folder in your Drive first, '
    f'with dataset/ and prompts.json inside it.'
)
print(f'Project dir OK: {PROJECT_DIR}')

# --- 2. Verify GPU ---
import torch
assert torch.cuda.is_available(), 'No GPU detected -- go to Runtime > Change runtime type > GPU.'
print(f'GPU OK: {torch.cuda.get_device_name(0)}')

# --- 3. Install lightweight deps (fast every time) ---
!pip install -q mediapipe opencv-python

# --- 4. Clone + install SAM2 (ephemeral, rebuilt every session -- fast) ---
if not os.path.isdir('/content/sam2_repo'):
    !git clone -q https://github.com/facebookresearch/sam2.git /content/sam2_repo
%cd /content/sam2_repo
!pip install -q -e .
%cd /content

# --- 5. Clone + install CoTracker (ephemeral, rebuilt every session) ---
if not os.path.isdir('/content/cotracker_repo'):
    !git clone -q https://github.com/facebookresearch/co-tracker.git /content/cotracker_repo
%cd /content/cotracker_repo
!pip install -q -e .
%cd /content

# --- 6. Download checkpoints ONLY if not already cached in Drive ---
sam2_ckpt_dir = f'{PROJECT_DIR}/checkpoints'
sam2_ckpt = f'{sam2_ckpt_dir}/sam2.1_hiera_small.pt'
os.makedirs(sam2_ckpt_dir, exist_ok=True)
if not os.path.isfile(sam2_ckpt):
    print('Downloading SAM2 checkpoint (first time only)...')
    !wget -q -O {sam2_ckpt} https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt
else:
    print('SAM2 checkpoint already cached in Drive, skipping download.')

cotracker_ckpt_dir = f'{PROJECT_DIR}/cotracker_checkpoints'
cotracker_ckpt = f'{cotracker_ckpt_dir}/scaled_offline.pth'
os.makedirs(cotracker_ckpt_dir, exist_ok=True)
if not os.path.isfile(cotracker_ckpt):
    print('Downloading CoTracker checkpoint (first time only)...')
    !wget -q -O {cotracker_ckpt} https://huggingface.co/facebook/cotracker3/resolve/main/scaled_offline.pth
else:
    print('CoTracker checkpoint already cached in Drive, skipping download.')

# --- 7. Copy SAM2 configs into Drive ONLY if not already there ---
configs_dir = f'{PROJECT_DIR}/configs'
if not os.path.isdir(configs_dir):
    print('Copying SAM2 configs into Drive (first time only)...')
    !cp -r /content/sam2_repo/sam2/configs {configs_dir}
else:
    print('Configs already cached in Drive, skipping copy.')

# --- 8. Make sure your pipeline scripts are present in Drive ---
scripts_dir = f'{PROJECT_DIR}/scripts'
os.makedirs(scripts_dir, exist_ok=True)
required_scripts = [
    'extract_frames.py', 'segment_video.py', 'extract_pose.py',
    'sample_seed_points.py', 'track_points.py', 'qc_trajectories.py',
    'visualize_frames.py',
]
missing = [s for s in required_scripts if not os.path.isfile(f'{scripts_dir}/{s}')]
if missing:
    print(f'WARNING: missing scripts in {scripts_dir}: {missing}')
    print('Upload them once via the Colab file browser (left sidebar) into this Drive folder.')
else:
    print('All pipeline scripts found in Drive.')

print('\n=== Setup complete ===')
print(f'PROJECT_DIR = {PROJECT_DIR}')
print('You can now run the pipeline cells below, in order, whenever you like this session.')

## Cell 2: Phase 1 -- Frame extraction (30fps)

In [ ]:
!python {PROJECT_DIR}/scripts/extract_frames.py \
  --input {PROJECT_DIR}/dataset \
  --output {PROJECT_DIR}/frames \
  --fps 30 --width 512

## Cell 3: Phase 2 -- SAM2 garment segmentation

In [ ]:
!python {PROJECT_DIR}/scripts/segment_video.py \
  --frames {PROJECT_DIR}/frames \
  --prompts {PROJECT_DIR}/prompts.json \
  --output {PROJECT_DIR}/masks \
  --checkpoint {PROJECT_DIR}/checkpoints/sam2.1_hiera_small.pt \
  --config {PROJECT_DIR}/configs/sam2.1/sam2.1_hiera_s.yaml

## Cell 4: Phase 2 -- MediaPipe pose extraction

In [ ]:
!python {PROJECT_DIR}/scripts/extract_pose.py \
  --frames {PROJECT_DIR}/frames \
  --output {PROJECT_DIR}/pose

## Cell 5: Phase 3, step 1 -- Seed point sampling (with re-seeding)

In [ ]:
!python {PROJECT_DIR}/scripts/sample_seed_points.py \
  --masks {PROJECT_DIR}/masks \
  --output {PROJECT_DIR}/seed_points \
  --grid-size 8 --reseed-interval 15

## Cell 6: Phase 3, step 2 -- CoTracker point tracking

In [ ]:
!python {PROJECT_DIR}/scripts/track_points.py \
  --frames {PROJECT_DIR}/frames \
  --seed-points {PROJECT_DIR}/seed_points \
  --output {PROJECT_DIR}/trajectories \
  --checkpoint {PROJECT_DIR}/cotracker_checkpoints/scaled_offline.pth

## Cell 7: QC check

In [ ]:
!python {PROJECT_DIR}/scripts/qc_trajectories.py \
  --trajectories {PROJECT_DIR}/trajectories \
  --masks {PROJECT_DIR}/masks \
  --frames {PROJECT_DIR}/frames \
  --output {PROJECT_DIR}/qc_report.json

## Cell 8 (optional): Visual review of a specific video

In [ ]:
VIDEO_TO_REVIEW = 'real/Man Excersing'  # change this to whichever video you want to inspect

!python {PROJECT_DIR}/scripts/visualize_frames.py \
  --frames {PROJECT_DIR}/frames \
  --trajectories {PROJECT_DIR}/trajectories \
  --output {PROJECT_DIR}/frame_review \
  --video "{VIDEO_TO_REVIEW}" --step 5

---
## Notes

- Everything under `PROJECT_DIR` (Drive) persists automatically -- no manual downloading/uploading between sessions.
- To browse results directly, use the Colab file browser (folder icon, left sidebar) and navigate into `drive/MyDrive/FYP/`.
- If you update any script locally, just re-upload that one file into `MyDrive/FYP/scripts/` (overwrite), no need to redo Cell 1's installs.
- `annotate_prompts.py` is NOT run here -- it needs a local display window. Run it on your laptop, then upload the resulting `prompts.json` into `MyDrive/FYP/` once.